# Gemma 4 31B Instruct — Kaggle TPU v5e-8 Text + Vision Production Showcase

### English

This notebook is a **live end-to-end production showcase**, not a startup smoke test. It loads one Gemma 4 31B Instruct model across **8 TPU devices**, starts the authenticated REST service, and sends real text and vision requests through the same public client/API path an application would use.

#### What this notebook proves

- English technical Q&A
- Vietnamese generation quality
- structured instruction following
- Python code generation
- identical-request hot executable reuse
- image-conditioned text generation
- spatial grounding and visible-text reading
- per-request latency and compile evidence

#### How to read this notebook

Every executable cell is preceded by an explanation covering:

1. **Purpose** — why the step exists.
2. **What the cell does** — the main actions performed.
3. **Expected result** — what a successful run should show.
4. **Interpretation** — what the result means for a production deployment.

#### Runtime architecture

```text
Kaggle Notebook
      │
      ├── TPU preflight ───────────────► 8 × TPU devices
      │
      ├── start.sh ─► REST coordinator ─► long-lived TPU worker
      │                                      │
      │                                      └── Gemma 4 31B Instruct
      │
      └── Python client ─► async REST jobs ─► text / vision responses
```

The long-lived worker is important: model loading and first-time compilation are expensive, while compatible later requests can reuse compiled executables.

> **Performance note:** a previous qualification run measured about **541.4 s** for the first compatible request and **5.24 s** for an identical hot request. This notebook always measures the current run again; those historical numbers are evidence, not an SLA.

#### Before you run

1. Select **TPU v5e-8 / v5litepod-8** in Kaggle session options.
2. Attach the **Gemma 4 31B Keras preset** as a Kaggle input.
3. Enable Internet so the notebook can access the repository and install dependencies when needed.
4. Choose **Run All** and keep the session alive while the model loads.

The notebook generates random per-session API secrets and never prints their values.

---

### Tiếng Việt

Notebook này là một **production showcase end-to-end thực tế**, không chỉ là bài kiểm tra khởi động. Notebook load một Gemma 4 31B Instruct trên **8 TPU devices**, khởi động REST service có xác thực và gửi các request text/vision thật qua đúng public client/API mà ứng dụng bên ngoài sẽ sử dụng.

#### Notebook này chứng minh điều gì?

- Hỏi đáp kỹ thuật bằng tiếng Anh
- Khả năng trả lời tiếng Việt
- Tuân thủ định dạng và yêu cầu có cấu trúc
- Sinh mã Python
- Tái sử dụng executable cho request giống hệt
- Hiểu ảnh và sinh văn bản từ ảnh + prompt
- Hiểu quan hệ không gian và đọc chữ trong ảnh
- Đo latency và compile evidence cho từng request

#### Cách đọc notebook

Mỗi cell thực thi đều có phần giải thích theo bốn ý:

1. **Mục đích** — vì sao bước này cần thiết.
2. **Cell làm gì** — các thao tác chính.
3. **Kết quả mong đợi** — dấu hiệu của một lần chạy thành công.
4. **Cách diễn giải** — ý nghĩa đối với triển khai production.

#### Kiến trúc runtime

```text
Kaggle Notebook
      │
      ├── TPU preflight ───────────────► 8 × TPU devices
      │
      ├── start.sh ─► REST coordinator ─► long-lived TPU worker
      │                                      │
      │                                      └── Gemma 4 31B Instruct
      │
      └── Python client ─► async REST jobs ─► text / vision responses
```

Worker được giữ sống lâu dài vì load model và compile lần đầu rất tốn thời gian, trong khi các request tương thích về sau có thể tái sử dụng executable đã compile.

> **Lưu ý hiệu năng:** một qualification run trước đây đo khoảng **541,4 giây** cho request tương thích đầu tiên và **5,24 giây** cho identical hot request. Notebook này luôn đo lại ở lần chạy hiện tại; các số lịch sử chỉ là bằng chứng, không phải SLA.

#### Trước khi chạy

1. Chọn **TPU v5e-8 / v5litepod-8** trong phần cấu hình session của Kaggle.
2. Attach **Gemma 4 31B Keras preset** vào notebook dưới dạng Kaggle input.
3. Bật Internet để notebook có thể truy cập repository và cài dependency khi cần.
4. Chọn **Run All** và giữ session hoạt động trong lúc model được load.

Notebook tự sinh API secret ngẫu nhiên cho mỗi session và không in giá trị secret ra output.


### English

## 1. Session configuration

### Purpose
Define every path, endpoint and showcase-wide constant in one place so the run is reproducible and easy to audit.

### What this cell does
- Selects the official GitHub repository and local workspace
- Sets the Gemma 4 31B preset name and local REST URL
- Defines the shared output budget of 96 new tokens
- Defines paths for the result JSON and deterministic vision card
- Creates a small shell helper used by later cells

### Expected result
You should see the selected model preset and detected Kaggle session type. No TPU work happens yet.

---

### Tiếng Việt

Cấu hình phiên chạy

Mục đích
Định nghĩa toàn bộ đường dẫn, endpoint và hằng số dùng chung tại một nơi để lần chạy có thể tái lập và dễ kiểm tra.

Cell này làm gì
Chọn repository GitHub chính thức và workspace local.
Đặt tên preset Gemma 4 31B và REST URL local.
Đặt budget output chung là 96 new tokens.
Định nghĩa đường dẫn JSON kết quả và diagnostic vision card.
Tạo shell helper dùng cho các cell sau.

Kết quả mong đợi
Bạn sẽ thấy model preset đã chọn và loại Kaggle session được phát hiện. Bước này chưa thực hiện công việc trên TPU.


In [ ]:
import json, os, secrets, subprocess, sys, time
from pathlib import Path
from statistics import mean
from IPython.display import Markdown, display

REPO_URL = 'https://github.com/dangkhoa2016/KerasHub-Gemma4-31B-IT-Kaggle-TPU-v5e8-Text-Vision.git'
REPO_DIR = Path('/kaggle/working/gemma4-31b-tpu')
MODEL_PRESET = 'gemma4_instruct_31b'
BASE_URL = 'http://127.0.0.1:7860'
SHOWCASE_MAX_NEW_TOKENS = 96
RESULTS_JSON = Path('/kaggle/working/gemma4-showcase-results.json')
TEST_CARD = Path('/kaggle/working/gemma4-vision-test-card.png')

PUBLIC_FILES = (
    'scripts/configure_kaggle_tpu.sh',
    'scripts/check_tpu_preflight.py',
    'scripts/resolve_kaggle_model_path.py',
    'scripts/ensure_libtpu.py',
    'scripts/start.sh',
    'scripts/status.sh',
    'scripts/stop.sh',
    'scripts/wait_ready.py',
    'clients/python/gemma4_client.py',
    '.env.example',
)

def sh(args, **kwargs):
    print('$', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], **kwargs)

print('MODEL_PRESET=' + MODEL_PRESET)
print('SESSION=' + os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'not-kaggle'))


### English

## 2. Check out the public source

### Purpose
Bind the demo to an exact Git commit so the implementation that produced the outputs is unambiguous.

### What this cell does
The cell clones or refreshes the repository, checks out origin/main, records SERVER_SHA and verifies every public script, client and showcase asset required by the notebook.

### Expected result
- One exact SERVER_SHA
- PUBLIC_FILES_PRESENT with the expected count
- No missing-public-artifact error

### Why it matters
The Git SHA is the provenance anchor connecting notebook outputs, source code and the public release.

---

### Tiếng Việt

Lấy đúng mã nguồn công khai

Mục đích
Gắn demo với một Git commit cụ thể để phiên bản implementation tạo ra output là hoàn toàn rõ ràng.

Cell này làm gì
Cell clone hoặc refresh repository, checkout origin/main, ghi lại SERVER_SHA và xác minh mọi script, client và showcase asset công khai mà notebook cần.

Kết quả mong đợi
Một SERVER_SHA chính xác.
PUBLIC_FILES_PRESENT với số lượng mong đợi.
Không có lỗi thiếu public artifact.

Vì sao quan trọng
Git SHA là mốc provenance liên kết output notebook với source code và public release.


In [ ]:
if not (REPO_DIR / '.git').exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    sh(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sh(['git', '-C', REPO_DIR, 'fetch', '--prune', 'origin'], check=True)
sh(['git', '-C', REPO_DIR, 'checkout', 'origin/main'], check=True)
sh(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)

SERVER_SHA = subprocess.check_output(
    ['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True
).strip()
missing = [name for name in PUBLIC_FILES if not (REPO_DIR / name).is_file()]
assert not missing, f'Public repository files are missing: {missing}'
print('SERVER_SHA=' + SERVER_SHA)
print('PUBLIC_FILES_PRESENT=' + str(len(PUBLIC_FILES)))


### English

## 3. Install the runtime

### Purpose
Align the Python application stack with the versions expected by the qualified TPU runtime.

### What this cell does
It installs requirements.txt and verifies the pinned libtpu runtime. Kaggle supplies the accelerator-aware JAX/JAXLIB base; the project installs the remaining application dependencies around it.

### Expected result
Package installation should finish successfully and ensure_libtpu.py should exit with code 0.

### Interpretation
A dependency-installation PASS proves software consistency only; TPU availability is verified separately afterward.

---

### Tiếng Việt

Cài runtime

Mục đích
Đồng bộ Python application stack với các phiên bản mà TPU runtime đã qualification yêu cầu.

Cell này làm gì
Cell cài requirements.txt và xác minh libtpu đã pin. Kaggle cung cấp JAX/JAXLIB base có nhận biết accelerator; project cài các application dependency còn lại.

Kết quả mong đợi
Quá trình cài package phải hoàn tất thành công và ensure_libtpu.py phải thoát với code 0.

Cách diễn giải
Dependency-installation PASS chỉ chứng minh tính nhất quán của software; TPU thật được xác minh riêng ở bước sau.


In [ ]:
sh([
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--root-user-action=ignore', '--no-warn-conflicts', '--progress-bar', 'off',
    '-r', REPO_DIR / 'requirements.txt'
], cwd=REPO_DIR, check=True)

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR / 'src')
sh([sys.executable, 'scripts/ensure_libtpu.py'], cwd=REPO_DIR, env=env, check=True)


### English

## 4. Resolve the attached Gemma 4 model

### Purpose
Locate the Kaggle-mounted Gemma 4 31B Keras preset without hard-coding a user-specific input path.

### What this cell does
It searches /kaggle/input, asks the public resolver for the preset path and verifies that the returned directory exists.

### Expected result
A valid path containing gemma4_instruct_31b is printed.

### If it fails
Attach the Gemma 4 31B Keras model input and rerun the cell; do not modify server code to bypass a missing model.

---

### Tiếng Việt

Xác định model Gemma 4 đã attach

Mục đích
Tìm Gemma 4 31B Keras preset được Kaggle mount mà không hard-code đường dẫn input riêng của từng tài khoản.

Cell này làm gì
Cell tìm trong /kaggle/input, gọi public resolver để lấy preset path và xác minh thư mục trả về tồn tại.

Kết quả mong đợi
Một đường dẫn hợp lệ có gemma4_instruct_31b sẽ được in ra.

Nếu thất bại
Attach đúng Gemma 4 31B Keras model input rồi chạy lại cell; không sửa server code để bỏ qua lỗi thiếu model.


In [ ]:
resolved = sh([
    sys.executable, 'scripts/resolve_kaggle_model_path.py',
    '--input-root', '/kaggle/input', '--preset-name', MODEL_PRESET
], cwd=REPO_DIR, env=env, text=True, capture_output=True)
if resolved.returncode != 0:
    print(resolved.stdout, end='')
    print(resolved.stderr, end='', file=sys.stderr)
    raise SystemExit('Attach the Gemma 4 31B model dataset and run this cell again.')

MODEL_PATH = resolved.stdout.strip().splitlines()[-1].strip()
assert Path(MODEL_PATH).exists(), f'Model path does not exist: {MODEL_PATH}'
print('MODEL_PATH=' + MODEL_PATH)


### English

## 5. Configure secrets and TPU discovery

### Purpose
Prepare a production-like authenticated service while keeping credentials private, and configure Kaggle TPU topology fallback variables before JAX probes the accelerator.

### What this cell does
- Generates random API_KEY and RESTART_SECRET values
- Writes them to .env without printing their values
- Inserts the resolved MODEL_PATH
- Sources configure_kaggle_tpu.sh and imports its environment back into Python
- Records effective TPU_ACCELERATOR_TYPE and TPU_SKIP_MDS_QUERY values

### Why the environment import matters
A shell subprocess cannot mutate the parent Python process environment. Without importing the exported values back, JAX may incorrectly query cloud metadata and stall.

### Expected result
TPU_CONFIG_IMPORTED=PASS, accelerator type v5e-8 and metadata-query skip enabled.

---

### Tiếng Việt

Cấu hình secret và TPU discovery

Mục đích
Chuẩn bị service có xác thực giống production nhưng không làm lộ credential, đồng thời cấu hình các biến topology fallback của Kaggle trước khi JAX kiểm tra accelerator.

Cell này làm gì
Sinh API_KEY và RESTART_SECRET ngẫu nhiên.
Ghi chúng vào .env nhưng không in giá trị.
Đưa MODEL_PATH đã resolve vào cấu hình.
Source configure_kaggle_tpu.sh rồi import environment trở lại Python.
Ghi lại giá trị TPU_ACCELERATOR_TYPE và TPU_SKIP_MDS_QUERY có hiệu lực.

Vì sao phải import environment
Shell subprocess không thể thay đổi environment của Python process cha. Nếu không import ngược các biến đã export, JAX có thể truy vấn cloud metadata sai và bị treo.

Kết quả mong đợi
Cần thấy TPU_CONFIG_IMPORTED=PASS, accelerator type v5e-8 và chế độ bỏ qua metadata query được bật.


In [ ]:
keys = {
    'API_KEY': secrets.token_urlsafe(24),
    'RESTART_SECRET': secrets.token_urlsafe(24),
}
lines = []
for line in (REPO_DIR / '.env.example').read_text(encoding='utf-8').splitlines():
    name = line.split('=', 1)[0]
    if name in keys:
        lines.append(f'{name}={keys[name]}')
    elif name == 'MODEL_PATH':
        lines.append('MODEL_PATH=' + MODEL_PATH)
    else:
        lines.append(line)
(REPO_DIR / '.env').write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('ENV_WRITTEN=.env (secrets generated, not printed)')
# Make the freshly written .env values available to subprocess-based preflight.
for raw in (REPO_DIR / '.env').read_text(encoding='utf-8').splitlines():
    if raw and not raw.lstrip().startswith('#') and '=' in raw:
        name, value = raw.split('=', 1)
        env[name] = value

# configure_kaggle_tpu.sh exports fallback topology variables. Because a plain
# subprocess cannot mutate this Python process environment, source it and import
# the resulting environment back into `env` before the preflight subprocess.
configured = subprocess.run(
    ['bash', '-lc', 'source scripts/configure_kaggle_tpu.sh >&2; env -0'],
    cwd=REPO_DIR,
    env=env,
    capture_output=True,
    check=True,
)
for item in configured.stdout.decode().split('\0'):
    if '=' in item:
        name, value = item.split('=', 1)
        env[name] = value
print('TPU_CONFIG_IMPORTED=PASS')
print('TPU_ACCELERATOR_TYPE=' + env.get('TPU_ACCELERATOR_TYPE', ''))
print('TPU_SKIP_MDS_QUERY=' + env.get('TPU_SKIP_MDS_QUERY', ''))


### English

## 6. TPU preflight — prove all 8 devices are visible

### Purpose
Fail early before loading tens of gigabytes of model weights if the Kaggle session is not connected to the expected TPU topology.

### What this cell does
The public preflight script initializes JAX TPU discovery and validates the expected accelerator and device count.

### Expected result
You should see JAX information and 8 TPU devices, followed by TPU_PREFLIGHT=PASS.

### Interpretation
This is the hardware gate. If it fails, model-load or inference results from the same session should not be trusted.

---

### Tiếng Việt

TPU preflight — xác nhận đủ 8 devices

Mục đích
Dừng sớm trước khi load hàng chục GB weights nếu session Kaggle không thật sự có đúng TPU topology.

Cell này làm gì
Public preflight script khởi tạo JAX TPU discovery và xác minh đúng accelerator cùng số lượng device mong đợi.

Kết quả mong đợi
Bạn cần thấy thông tin JAX và 8 TPU devices, sau đó là TPU_PREFLIGHT=PASS.

Cách diễn giải
Đây là hardware gate. Nếu bước này fail thì không nên tin cậy kết quả model-load hoặc inference của cùng session.


In [ ]:
preflight = sh(
    [sys.executable, 'scripts/check_tpu_preflight.py'],
    cwd=REPO_DIR, env=env, text=True, capture_output=True,
)
print(preflight.stdout, end='')
if preflight.returncode != 0:
    print(preflight.stderr, end='', file=sys.stderr)
    raise SystemExit('TPU preflight failed. Enable TPU v5e-8 and restart the session.')
print('TPU_PREFLIGHT=PASS')


### English

## 7. Start one long-lived server

### Purpose
Load the 31B model once and keep one TPU worker alive across multiple requests so compatible later requests can reuse runtime state and compiled executables.

### What this cell does
- Starts the REST coordinator and TPU worker with scripts/start.sh
- Waits on /health/ready for up to one hour
- Measures total server startup time
- Prints the tail of the server log if readiness fails

### What to expect
This is usually the longest setup stage because it includes TPU initialization, model construction, sharding and checkpoint loading. A long startup is expected; repeated model loads per request are not.

### Successful marker
SERVER_READY=PASS with the measured number of seconds.

---

### Tiếng Việt

Khởi động server dài hạn

Mục đích
Load model 31B một lần và giữ một TPU worker sống xuyên suốt nhiều request để các request tương thích về sau có thể tái sử dụng runtime state và executable đã compile.

Cell này làm gì
Khởi động REST coordinator và TPU worker bằng scripts/start.sh.
Chờ /health/ready tối đa một giờ.
Đo tổng thời gian server startup.
In phần cuối server log nếu readiness thất bại.

Điều cần chờ
Đây thường là bước setup lâu nhất do phải khởi tạo TPU, dựng model, sharding và load checkpoint. Startup lâu là bình thường; load lại model cho từng request thì không bình thường.

Dấu hiệu thành công
Dấu hiệu thành công là SERVER_READY=PASS kèm số giây đo được.


In [ ]:
startup_started = time.perf_counter()
sh(['bash', 'scripts/start.sh'], cwd=REPO_DIR, env=env, check=True)
ready_env = dict(env, READY_URL=BASE_URL + '/health/ready', READY_TIMEOUT='3600')
waited = sh(
    [sys.executable, 'scripts/wait_ready.py'], cwd=REPO_DIR,
    env=ready_env, text=True, capture_output=True,
)
if waited.returncode != 0:
    log_path = REPO_DIR / 'logs' / 'server.stdout.log'
    if log_path.exists():
        print(log_path.read_text(encoding='utf-8', errors='replace')[-5000:])
    raise SystemExit('Server did not become ready.')
SERVER_STARTUP_SECONDS = time.perf_counter() - startup_started
print(f'SERVER_READY=PASS ({SERVER_STARTUP_SECONDS:.2f} s)')


### English

## 8. Inspect the live runtime

### Purpose
Show what is actually serving requests before the demo starts: model identity, backend, device count and production-generation metadata.

### What this cell does
It calls the authenticated /info endpoint using the generated API key and prints the returned JSON.

### Expected result
The metadata should correspond to Gemma 4 31B, JAX, the expected TPU topology and the production generation path.

### Why it matters
This separates a live server process from confirmation that the intended model and runtime configuration are actually active.

---

### Tiếng Việt

Kiểm tra runtime đang chạy

Mục đích
Hiển thị runtime thực tế đang phục vụ request trước khi demo: model identity, backend, device count và production-generation metadata.

Cell này làm gì
Cell gọi endpoint /info có xác thực bằng API key vừa sinh và in JSON trả về.

Kết quả mong đợi
Metadata phải tương ứng với Gemma 4 31B, JAX, TPU topology mong đợi và production generation path.

Vì sao quan trọng
Bước này phân biệt giữa việc server process đang sống và việc đúng model/runtime mong muốn thật sự đang hoạt động.


In [ ]:
import urllib.request

settings = dict(
    line.split('=', 1) for line in
    (REPO_DIR / '.env').read_text(encoding='utf-8').splitlines()
    if '=' in line
)
API_KEY = settings['API_KEY']

req = urllib.request.Request(
    BASE_URL + '/info', headers={'Authorization': f'Bearer {API_KEY}'}
)
with urllib.request.urlopen(req, timeout=30) as response:
    INFO = json.loads(response.read().decode())

display(Markdown('### Live service metadata'))
print(json.dumps(INFO, indent=2, ensure_ascii=False))


### English

## 9. Showcase harness — timing, output and compile evidence

### Purpose
Use one common request function so every text and vision case is measured and reported consistently.

### What this cell does
- Imports the shipped public Python client
- Sends requests through the public async REST contract
- Measures client wall-clock latency
- Records server-side inference time
- Extracts prompt tokens, generation bucket and compile-cache evidence
- Stores every case in SHOWCASE for the final table

### Important metric distinction
Wall time includes client polling, network and local coordination overhead. Inference time is reported by the server for model execution.

### Expected result
This cell only defines helpers and should not perform inference yet.

---

### Tiếng Việt

Bộ khung đo showcase

Mục đích
Dùng một hàm request chung để tất cả case text/vision được đo và báo cáo nhất quán.

Cell này làm gì
Import public Python client đi kèm project.
Gửi request qua public async REST contract.
Đo wall-clock latency phía client.
Ghi inference time phía server.
Trích prompt tokens, generation bucket và compile-cache evidence.
Lưu mọi case vào SHOWCASE để tạo bảng cuối.

Phân biệt metric quan trọng
Wall time gồm polling, network và local coordination. Inference time là thời gian model execution do server báo cáo.

Kết quả mong đợi
Cell này chỉ định nghĩa helper và chưa thực hiện inference.


In [ ]:
sys.path.insert(0, str(REPO_DIR / 'clients' / 'python'))
from gemma4_client import Gemma4Client

client = Gemma4Client(BASE_URL, API_KEY, request_timeout=30, poll_timeout=3600)
SHOWCASE = []

def _compile_summary(metrics):
    ev = (metrics or {}).get('compile_cache_evidence') or {}
    return {
        'compile_attempts': ev.get('compile_attempt_count'),
        'compile_events': ev.get('compile_event_count'),
    }

def run_case(case, prompt, *, system='', image=None, max_new_tokens=SHOWCASE_MAX_NEW_TOKENS):
    display(Markdown(f'### {case}'))
    print('PROMPT:', prompt)
    started = time.perf_counter()
    if image is None:
        body = client.generate(prompt, system, max_new_tokens)
        modality = 'text'
    else:
        body = client.generate_image(image, prompt, system, max_new_tokens)
        modality = 'vision'
    wall = time.perf_counter() - started
    metrics = body.get('metrics') or {}
    output = body.get('output', '')
    comp = _compile_summary(metrics)
    row = {
        'case': case,
        'modality': modality,
        'wall_seconds': round(wall, 3),
        'inference_seconds': body.get('inference_seconds'),
        'prompt_tokens': metrics.get('prompt_tokens'),
        'max_new_tokens': metrics.get('max_new_tokens'),
        'generation_max_length': metrics.get('generation_max_length'),
        'compile_attempts': comp['compile_attempts'],
        'compile_events': comp['compile_events'],
        'vision_conditioning': metrics.get('vision_conditioning_present'),
        'output': output,
    }
    SHOWCASE.append(row)
    display(Markdown('**Response**'))
    print(output)
    print('\nTIMING:', json.dumps({k: row[k] for k in (
        'wall_seconds','inference_seconds','prompt_tokens','generation_max_length',
        'compile_attempts','compile_events','vision_conditioning'
    )}, ensure_ascii=False))
    return row


### English

# Text showcase

The next six requests exercise different practical behaviors while keeping the output budget at `96` new tokens. Short prompts are intentionally similar in size so compatible requests can often share the same production length bucket.

## What to watch

- The **first compatible request** may include compile cost.
- The **identical repeat** should demonstrate the strongest hot-reuse effect.
- Different prompts show that the service is not merely replaying cached text.
- Vietnamese and code tasks demonstrate output diversity, not benchmark accuracy scoring.

---

### Tiếng Việt

Trình diễn text

Sáu request tiếp theo kiểm tra nhiều hành vi thực tế khác nhau nhưng giữ cùng budget `96` new tokens. Các prompt ngắn được cố ý giữ kích thước tương tự để những request tương thích có cơ hội dùng chung production length bucket.

Điều cần quan sát

- **Request tương thích đầu tiên** có thể chịu compile cost.
- **Identical repeat** thường thể hiện hot-reuse rõ nhất.
- Prompt khác nhau chứng minh service không chỉ replay output cache.
- Tiếng Việt và code dùng để trình diễn độ đa dạng, không phải bài chấm điểm benchmark.


### English

## 10. Technical Q&A — first compatible text request

### Purpose
Establish first-request latency for this text generation shape while checking whether the model can produce a concise technical explanation.

### Prompt behavior
The model is asked to explain why model parallelism is useful for a 31B-parameter model spread across eight TPU devices.

### Expected result
A coherent three-sentence explanation plus timing and compile metadata. The first request can be much slower because compilation may happen here.

---

### Tiếng Việt

Hỏi đáp kỹ thuật — request text tương thích đầu tiên

Mục đích
Thiết lập latency của request đầu tiên cho text generation shape này, đồng thời kiểm tra model có thể đưa ra giải thích kỹ thuật ngắn gọn hay không.

Nội dung prompt
Model được yêu cầu giải thích vì sao model parallelism hữu ích với model 31B parameters được phân bố trên tám TPU devices.

Kết quả mong đợi
Cần có giải thích mạch lạc trong ba câu kèm timing và compile metadata. Request đầu tiên có thể chậm hơn nhiều vì compilation có thể xảy ra tại đây.


In [ ]:
TECH_PROMPT = 'In three sentences, explain why model parallelism is useful for a 31B-parameter model on eight TPU devices.'
tech_first = run_case(
    'Text 1 — technical explanation (first)',
    TECH_PROMPT,
    system='Be technically accurate, concise, and practical.',
)


### English

## 11. Identical hot repeat — executable reuse

### Purpose
Demonstrate the central production optimization: the exact same system prompt, user prompt and token budget are sent again immediately.

### What to compare
Compare `inference_seconds` from Text 1 and Text 2. The notebook prints `IDENTICAL_REQUEST_SPEEDUP` when both measurements are available.

### Interpretation
A large speedup indicates executable/runtime reuse for the compatible hot request. It does **not** mean every future prompt will have the same latency; different shapes/buckets may compile separately.

---

### Tiếng Việt

Lặp lại request giống hệt — tái sử dụng executable

Mục đích

Trình diễn tối ưu quan trọng nhất: gửi lại ngay lập tức cùng system prompt, user prompt và token budget.

Cần so sánh gì

So sánh `inference_seconds` của Text 1 và Text 2. Notebook sẽ in `IDENTICAL_REQUEST_SPEEDUP` khi có đủ hai phép đo.

Cách diễn giải
Speedup lớn cho thấy hot request tương thích tái sử dụng executable/runtime. Điều này **không** có nghĩa mọi prompt khác đều có cùng latency.


In [ ]:
tech_hot = run_case(
    'Text 2 — identical hot repeat',
    TECH_PROMPT,
    system='Be technically accurate, concise, and practical.',
)
if tech_first.get('inference_seconds') and tech_hot.get('inference_seconds'):
    speedup = tech_first['inference_seconds'] / tech_hot['inference_seconds']
    print(f'IDENTICAL_REQUEST_SPEEDUP={speedup:.2f}x')


### English

## 12. Vietnamese response

### Purpose
Show that the same production endpoint can handle a natural Vietnamese technical prompt, not only English examples.

### Task
The model explains the difference between model parallelism and data parallelism with an easy-to-understand example, entirely in Vietnamese.

### What to evaluate visually
Language consistency, clarity, technical relevance and whether the response stays in Vietnamese as instructed.

---

### Tiếng Việt

Trả lời tiếng Việt

Mục đích
Chứng minh cùng production endpoint có thể xử lý prompt kỹ thuật tự nhiên bằng tiếng Việt, không chỉ tiếng Anh.

Bài toán
Model giải thích sự khác nhau giữa model parallelism và data parallelism bằng một ví dụ dễ hiểu, hoàn toàn bằng tiếng Việt.

Người xem nên quan sát
Quan sát tính nhất quán ngôn ngữ, độ rõ ràng, tính đúng chủ đề và việc model có tuân thủ yêu cầu trả lời hoàn toàn bằng tiếng Việt hay không.


In [ ]:
run_case(
    'Text 3 — Vietnamese',
    'Giải thích ngắn gọn sự khác nhau giữa model parallelism và data parallelism, kèm một ví dụ dễ hiểu.',
    system='Trả lời hoàn toàn bằng tiếng Việt, rõ ràng và súc tích.',
)


### English

## 13. Code generation

### Purpose
Demonstrate a common developer use case through the same deployed REST stack.

### Task
Generate a compact Python function that splits a list into N nearly equal contiguous shards and include one example call.

### What to inspect
The response should contain runnable-looking Python, respect the requested compactness and include an example without changing serving configuration.

---

### Tiếng Việt

Sinh mã nguồn

Mục đích
Trình diễn một use case phổ biến của lập trình viên thông qua cùng REST stack đang chạy.

Bài toán
Sinh một hàm Python gọn để chia list thành N shard liên tiếp có kích thước gần bằng nhau và kèm một ví dụ gọi hàm.

Cần xem gì
Response nên chứa Python code có thể chạy, gọn đúng yêu cầu và có ví dụ, trong khi server/runtime vẫn giữ nguyên.


In [ ]:
run_case(
    'Text 4 — Python code',
    'Write a small Python function that splits a list into N nearly equal contiguous shards. Include one example call.',
    system='Return compact, correct Python with only a short explanation.',
)


### English

## 14. Structured instruction following

### Purpose
Test whether the model follows a strict output-format instruction rather than only answering the topic correctly.

### Task
Return exactly three bullet points describing practical benefits of keeping a compiled JAX executable hot.

### What to inspect
Count the bullets and check that the response avoids an unnecessary introduction.

---

### Tiếng Việt

Tuân thủ yêu cầu có cấu trúc

Mục đích
Kiểm tra model có tuân thủ định dạng output chặt chẽ hay chỉ trả lời đúng nội dung.

Bài toán
Trả về đúng ba bullet points mô tả lợi ích thực tế của việc giữ JAX executable đã compile ở trạng thái hot.

Cần xem gì
Đếm đúng ba bullet và kiểm tra response không thêm phần mở đầu thừa.


In [ ]:
run_case(
    'Text 5 — structured answer',
    'Give exactly three bullet points describing practical benefits of keeping a compiled JAX executable hot between compatible inference requests.',
    system='Follow the requested format exactly and avoid an introduction.',
)


### English

## 15. Engineering reasoning — deployment trade-off

### Purpose
Show a short production-oriented reasoning response instead of another factual lookup.

### Task
Compare restarting the TPU worker after every request with keeping one long-lived worker, then choose the preferable design for this demo.

### Interpretation
The answer should connect startup, model-load and compile cost with service responsiveness.

---

### Tiếng Việt

Lập luận kỹ thuật — lựa chọn triển khai

Mục đích
Trình diễn một câu trả lời lập luận theo góc nhìn production thay vì chỉ hỏi kiến thức.

Bài toán
So sánh việc restart TPU worker sau mỗi request với giữ một worker sống lâu dài, rồi chọn thiết kế phù hợp cho demo này.

Cách diễn giải
Câu trả lời nên liên hệ chi phí startup, model-load và compile với độ phản hồi của service.


In [ ]:
run_case(
    'Text 6 — engineering trade-off',
    'A service can either restart its TPU worker after every request or keep one long-lived worker. Which design is preferable for this demo and why?',
    system='Answer as a production ML engineer in no more than four sentences.',
)


### English

# Vision showcase

Gemma 4 in this project performs **image-conditioned text generation**:

```text
image + text prompt  ──►  text response
```

```text
ảnh + prompt văn bản  ──►  câu trả lời văn bản
```

It does **not** generate PNG/JPEG images from text.

## Why the notebook creates its own test image

A deterministic visual card guarantees that every Run All includes a real vision test even when the user has not attached an image. The card contains shapes, colors, spatial relationships and readable text so multiple visual capabilities can be inspected independently.

---

### Tiếng Việt

Trình diễn Vision

```text
image + text prompt  ──►  text response
```

Trong dự án này, Gemma 4 thực hiện **image-conditioned text generation**:

```text
ảnh + prompt văn bản  ──►  câu trả lời văn bản
```

Nó **không** phải model text-to-image sinh PNG/JPEG từ prompt.

Vì sao notebook tự tạo ảnh test

Ảnh test xác định giúp mọi lần Run All đều có vision test thật dù người dùng chưa attach ảnh. Ảnh chứa hình khối, màu sắc, quan hệ không gian và chữ đọc được để kiểm tra nhiều khả năng thị giác khác nhau.


### English

## 16. Create a richer deterministic visual diagnostic card

### Purpose
Produce a known image with unambiguous ground truth so viewers can judge text reading, counting and spatial answers themselves.

### What this cell draws
- A red rounded rectangle at upper-left
- A blue circle at upper-right
- A green triangle at bottom-left
- A yellow star at bottom-right
- Two arrows with different directions
- A four-row key-value table
- Visible labels including GEMMA 4 VISION SHOWCASE 2026, 8 TPU DEVICES and TOTAL SHAPES = 4

### Expected result
The generated PNG is displayed directly in the notebook and saved under /kaggle/working.

---

### Tiếng Việt

Tạo diagnostic card trực quan phong phú hơn

Mục đích
Tạo ảnh có ground truth rõ ràng để người xem tự đối chiếu khả năng đọc chữ, đếm và hiểu quan hệ không gian.

Cell này vẽ gì
Hình chữ nhật bo góc màu đỏ ở phía trên-trái.
Hình tròn xanh ở phía trên-phải.
Tam giác xanh lá ở phía dưới-trái.
Ngôi sao vàng ở phía dưới-phải.
Hai mũi tên có hướng khác nhau.
Bảng key-value bốn dòng.
Các nhãn chữ rõ ràng tương ứng.

Kết quả mong đợi
Ảnh PNG được hiển thị ngay trong notebook và lưu dưới /kaggle/working.


In [ ]:
from PIL import Image, ImageDraw, ImageFont

W, H = 1400, 900
img = Image.new('RGB', (W, H), 'white')
draw = ImageDraw.Draw(img)

try:
    title_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 40)
    head_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 26)
    text_font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 22)
except OSError:
    title_font = head_font = text_font = ImageFont.load_default()

draw.text((40, 30), 'GEMMA 4 VISION SHOWCASE 2026', fill='black', font=title_font)
draw.text((40, 85), 'Read shapes, colors, positions, table entries, and visible text.', fill='dimgray', font=text_font)

draw.rounded_rectangle((90, 180, 360, 340), radius=28, fill='#d62828', outline='black', width=4)
draw.text((145, 245), 'A. RED BOX', fill='white', font=head_font)
draw.ellipse((980, 170, 1230, 420), fill='#2563eb', outline='black', width=4)
draw.text((1025, 275), 'B', fill='white', font=title_font)
draw.polygon([(220, 620), (120, 810), (320, 810)], fill='#16a34a', outline='black')
draw.text((165, 820), 'C. GREEN TRIANGLE', fill='black', font=text_font)

cx, cy = 1100, 700
star = [(cx, cy-90), (cx+25, cy-25), (cx+95, cy-25), (cx+38, cy+15),
        (cx+58, cy+85), (cx, cy+40), (cx-58, cy+85), (cx-38, cy+15),
        (cx-95, cy-25), (cx-25, cy-25)]
draw.polygon(star, fill='#facc15', outline='black')
draw.text((1010, 810), 'D. YELLOW STAR', fill='black', font=text_font)

draw.line((360, 260, 980, 260), fill='black', width=8)
draw.polygon([(980, 260), (940, 235), (940, 285)], fill='black')
draw.text((560, 225), 'Arrow 1 points right', fill='black', font=text_font)
draw.line((220, 620, 220, 360), fill='gray', width=6)
draw.polygon([(220, 360), (200, 395), (240, 395)], fill='gray')
draw.text((245, 470), 'Arrow 2 points up', fill='gray', font=text_font)

table_x, table_y = 500, 470
cell_w, cell_h = 270, 60
table_rows = [('LEFT OBJECT','RED BOX'),('RIGHT OBJECT','BLUE CIRCLE'),
              ('BOTTOM LEFT','GREEN TRIANGLE'),('BOTTOM RIGHT','YELLOW STAR')]
for r in range(4):
    for c in range(2):
        x1, y1 = table_x + c*cell_w, table_y + r*cell_h
        draw.rectangle((x1,y1,x1+cell_w,y1+cell_h), outline='black', width=2)
for i,(a,b) in enumerate(table_rows):
    draw.text((table_x+16, table_y+16+i*cell_h), a, fill='black', font=text_font)
    draw.text((table_x+cell_w+16, table_y+16+i*cell_h), b, fill='black', font=text_font)

draw.text((40, 860), 'VISIBLE TEXT CHECK: 8 TPU DEVICES', fill='#15803d', font=head_font)
draw.text((850, 860), 'TOTAL SHAPES = 4', fill='black', font=head_font)
img.save(TEST_CARD)
display(img)
print('TEST_CARD=' + str(TEST_CARD))


### English

## 17. Diagnostic vision — scene understanding

### Purpose
Test whether the model can combine object recognition, color recognition and visible-text understanding in one image.

### Task
Describe the card in three concise sentences, identify all four major shapes and colors, and mention the most prominent text.

### What to inspect
The answer should identify the red box, blue circle, green triangle and yellow star without inventing unrelated objects.

---

### Tiếng Việt

Vision diagnostic — hiểu toàn cảnh

Mục đích
Kiểm tra model có kết hợp được nhận diện vật thể, màu sắc và chữ nhìn thấy trong cùng một ảnh hay không.

Bài toán
Mô tả card trong ba câu ngắn gọn, xác định đủ bốn hình chính cùng màu sắc và nêu các dòng chữ nổi bật nhất.

Cần xem gì
Câu trả lời nên nhận đúng hộp đỏ, vòng tròn xanh, tam giác xanh lá và ngôi sao vàng mà không bịa thêm vật thể.


In [ ]:
vision_first = run_case(
    'Vision 1 — diagnostic scene understanding',
    'Describe this visual diagnostic card in three concise sentences. Identify all four major shapes, their colors, and the most prominent visible text.',
    system='Base the answer only on visible image content. Do not invent objects or text.',
    image=str(TEST_CARD),
)


### English

## 18. Diagnostic vision — spatial grounding and counting

### Purpose
Test where objects are located, the directions of arrows and whether the model can count the major shapes.

### Task
Identify the objects at left, right, bottom-left and bottom-right; state both arrow directions; report the total shape count.

### Expected result
The spatial answer should match the known card layout and report four major shapes.

---

### Tiếng Việt

Vision diagnostic — định vị không gian và đếm

Mục đích
Kiểm tra vị trí của vật thể, hướng mũi tên và khả năng đếm các hình chính.

Bài toán
Xác định vật thể ở trái, phải, dưới-trái và dưới-phải; nêu hướng của hai mũi tên; trả lời tổng số hình.

Kết quả mong đợi
Câu trả lời không gian phải khớp bố cục ground truth của card và nêu đúng bốn hình chính.


In [ ]:
vision_spatial = run_case(
    'Vision 2 — spatial grounding and counting',
    'Which object is on the left, right, bottom-left, and bottom-right? State the direction of both arrows and the total number of major shapes.',
    system='Answer only from the image and preserve spatial relationships accurately.',
    image=str(TEST_CARD),
)


### English

## 19. Diagnostic vision — visible text and table reading

### Purpose
Combine visual recognition with reading structured text embedded in the image.

### Task
Read the four rows of the central table, extract prominent footer text, and report the written TPU-device and shape counts.

### What to inspect
The result should preserve table associations and recover 8 TPU DEVICES plus TOTAL SHAPES = 4.

---

### Tiếng Việt

Vision diagnostic — đọc chữ và bảng

Mục đích
Kết hợp nhận diện hình ảnh với đọc nội dung chữ có cấu trúc nằm trong ảnh.

Bài toán
Đọc bốn dòng trong bảng trung tâm, trích các dòng footer nổi bật và trả lời số TPU devices cùng tổng số hình được ghi trên card.

Cần xem gì
Kết quả nên giữ đúng quan hệ trong bảng, đồng thời đọc được 8 TPU DEVICES và TOTAL SHAPES = 4.


In [ ]:
vision_text = run_case(
    'Vision 3 — visible text and table reading',
    'Read the four rows of the central table, list the prominent footer text, and state how many TPU devices and total shapes are written on the card.',
    system='Transcribe only text that is clearly visible; do not guess unreadable text.',
    image=str(TEST_CARD),
)


### English

## 20. Photorealistic portrait — natural scene understanding

### Image provenance
The image below is a synthetic photorealistic portrait generated by Mage-Flow-Turbo and committed with the public repository for reproducibility. It is not a camera photograph.

### Purpose
Move beyond geometric diagnostics and test a visually rich human-centered scene with weather, clothing, lighting and background context.

### Task
Describe the visible person, clothing, weather and environment in three or four concise sentences.

### What to inspect
Look for grounded details such as wet hair, layered outdoor clothing, backpack straps, rain and the blurred outdoor background; avoid identity guesses.

---

### Tiếng Việt

Chân dung photorealistic — hiểu cảnh tự nhiên

Nguồn ảnh
Ảnh bên dưới là chân dung photorealistic do Mage-Flow-Turbo sinh và được lưu cùng repository công khai để có thể tái lập. Đây không phải ảnh chụp camera.

Mục đích
Vượt ra ngoài hình học diagnostic để kiểm tra một cảnh có người với thời tiết, trang phục, ánh sáng và bối cảnh phong phú.

Bài toán
Mô tả người nhìn thấy trong ảnh, trang phục, thời tiết và môi trường xung quanh trong ba hoặc bốn câu ngắn gọn.

Cần xem gì
Quan sát xem model có bám đúng các chi tiết như tóc ướt, nhiều lớp áo ngoài trời, dây ba lô, mưa và background ngoài trời bị xóa phông hay không; không suy đoán danh tính.


In [ ]:
PORTRAIT_IMAGE = REPO_DIR / 'assets' / 'showcase' / 'mage-flow-portrait.png'
assert PORTRAIT_IMAGE.is_file(), f'Missing public portrait asset: {PORTRAIT_IMAGE}'
display(Image.open(PORTRAIT_IMAGE))

portrait_scene = run_case(
    'Vision 4 — photorealistic portrait scene',
    'Describe the visible person, clothing, weather, lighting, and environment in three or four concise sentences. Do not guess identity, nationality, ethnicity, or other hidden personal attributes.',
    system='Describe only visually supported details and clearly separate observation from interpretation.',
    image=str(PORTRAIT_IMAGE),
)


### English

## 21. Photorealistic portrait — detailed visual grounding

### Purpose
Test whether the model can answer focused questions about multiple visible attributes in the same natural-looking image.

### Task
Report the apparent weather, hair condition, outerwear, backpack visibility, dominant background colors and whether the background appears sharp or blurred.

### Interpretation
This is an attribute-grounding test, not face recognition. The notebook intentionally avoids asking the model to identify the person or infer sensitive traits.

---

### Tiếng Việt

Chân dung photorealistic — grounding chi tiết

Mục đích
Kiểm tra model có trả lời chính xác nhiều thuộc tính nhìn thấy trong cùng một ảnh tự nhiên hay không.

Bài toán
Nêu thời tiết nhìn thấy, trạng thái tóc, lớp áo ngoài, việc có nhìn thấy ba lô hay không, màu nền chủ đạo và background sắc nét hay bị xóa phông.

Cách diễn giải
Đây là bài kiểm tra grounding thuộc tính, không phải nhận diện khuôn mặt. Notebook cố ý không yêu cầu model xác định danh tính hoặc suy luận các thuộc tính nhạy cảm.


In [ ]:
portrait_detail = run_case(
    'Vision 5 — photorealistic portrait grounding',
    'Based only on the image, state: (1) the apparent weather, (2) whether the hair looks dry or wet, (3) the visible outerwear, (4) whether backpack straps are visible, (5) the dominant background colors, and (6) whether the background is sharp or blurred.',
    system='Use concise numbered points. Do not infer identity or sensitive personal attributes.',
    image=str(PORTRAIT_IMAGE),
)


### English

## 22. Optional — analyze your own image

### Purpose
Let viewers extend the demo to a real personal/test image without changing notebook code.

### How to enable
Set environment variable `SHOWCASE_IMAGE` to a `.jpg`, `.jpeg`, `.png` or `.webp` file under `/kaggle/input` before this cell runs.

### Important
This cell is optional. The five built-in vision tests above already exercise the vision path even when `SHOWCASE_IMAGE` is not supplied.

---

### Tiếng Việt

Tùy chọn — phân tích ảnh của bạn

Mục đích

Cho phép người xem thử thêm ảnh thật của mình mà không cần sửa code notebook.

Cách bật

Đặt biến môi trường `SHOWCASE_IMAGE` trỏ tới file ảnh dưới `/kaggle/input` trước khi chạy cell.

Quan trọng
Cell này là tùy chọn. Năm vision test tích hợp phía trên đã kiểm tra vision path ngay cả khi không cung cấp `SHOWCASE_IMAGE`.


In [ ]:
external = os.environ.get('SHOWCASE_IMAGE', '').strip()
if external:
    p = Path(external)
    if not p.is_file():
        raise FileNotFoundError(f'SHOWCASE_IMAGE does not exist: {p}')
    display(Image.open(p))
    run_case(
        'Vision 4 — user image',
        'Describe this image, identify the most important visible elements, and mention any readable text.',
        system='Be factual and concise.',
        image=str(p),
    )
else:
    print('OPTIONAL_USER_IMAGE=SKIPPED (set SHOWCASE_IMAGE to enable)')


### English

# Showcase summary

The inference work is complete. The remaining cells transform the collected measurements into a compact human-readable table and a machine-readable JSON artifact.

---

### Tiếng Việt

Tổng kết showcase

Phần inference đã hoàn tất. Các cell còn lại chuyển dữ liệu đo thành bảng dễ đọc cho con người và JSON machine-readable để audit/tái sử dụng.


### English

## 23. Latency table and run evidence

### Purpose
Provide one place where a reviewer can compare every request without scrolling through all previous outputs.

### Columns
- Wall s — total client-observed elapsed time
- Inference s — server-reported model execution time
- Prompt tokens — encoded prompt size
- Max length — production generation bucket or limit
- Compile attempts/events — compile-cache evidence

### Additional output
The cell writes /kaggle/working/gemma4-showcase-results.json with the same measurements and all responses in machine-readable form.

### Interpretation
Use Text 1 and Text 2 to discuss hot reuse. Do not treat average latency as an SLA.

---

### Tiếng Việt

Bảng latency và bằng chứng lần chạy

Mục đích
Cung cấp một bảng duy nhất để người review so sánh mọi request mà không phải cuộn lại toàn bộ notebook.

Ý nghĩa các cột
tổng thời gian phía client quan sát.
thời gian model execution do server báo cáo.
số token của prompt.
generation bucket hoặc giới hạn production.
bằng chứng compile-cache của request.

Output bổ sung
Cell ghi /kaggle/working/gemma4-showcase-results.json chứa cùng dữ liệu đo và toàn bộ response dưới dạng machine-readable.

Cách diễn giải
Dùng Text 1 và Text 2 để quan sát hot reuse. Không xem latency trung bình là SLA.


In [ ]:
def fmt(v):
    if v is None:
        return '—'
    return f'{v:.3f}' if isinstance(v, float) else str(v)

header = '| Case | Mode | Wall s | Inference s | Prompt tokens | Max length | Compile attempts | Compile events |\n|---|---:|---:|---:|---:|---:|---:|---:|'
rows = [header]
for r in SHOWCASE:
    rows.append('| {case} | {modality} | {wall} | {inf} | {pt} | {ml} | {ca} | {ce} |'.format(
        case=r['case'].replace('|','/'), modality=r['modality'], wall=fmt(r['wall_seconds']),
        inf=fmt(r['inference_seconds']), pt=fmt(r['prompt_tokens']), ml=fmt(r['generation_max_length']),
        ca=fmt(r['compile_attempts']), ce=fmt(r['compile_events'])
    ))
display(Markdown('\n'.join(rows)))

text_times = [r['inference_seconds'] for r in SHOWCASE if r['modality']=='text' and isinstance(r['inference_seconds'], (int,float))]
vision_times = [r['inference_seconds'] for r in SHOWCASE if r['modality']=='vision' and isinstance(r['inference_seconds'], (int,float))]
summary = {
    'server_sha': SERVER_SHA,
    'model_path': MODEL_PATH,
    'server_startup_seconds': round(SERVER_STARTUP_SECONDS, 3),
    'tpu_devices_expected': 8,
    'requests_completed': len(SHOWCASE),
    'text_requests': sum(r['modality']=='text' for r in SHOWCASE),
    'vision_requests': sum(r['modality']=='vision' for r in SHOWCASE),
    'mean_text_inference_seconds': round(mean(text_times), 3) if text_times else None,
    'mean_vision_inference_seconds': round(mean(vision_times), 3) if vision_times else None,
    'identical_text_speedup': (
        round(tech_first['inference_seconds']/tech_hot['inference_seconds'], 3)
        if tech_first.get('inference_seconds') and tech_hot.get('inference_seconds') else None
    ),
    'cases': SHOWCASE,
}
RESULTS_JSON.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k!='cases'}, indent=2, ensure_ascii=False))
print('RESULTS_JSON=' + str(RESULTS_JSON))


### English

## 24. Final server status

### Purpose
End the demo with an explicit operational check instead of assuming successful responses imply the service is still healthy.

### What this cell does
It calls the public status script, prints the final showcase markers and reports total completed requests.

### Successful end state
Look for SHOWCASE=PASS and COMPLETED_REQUESTS=11.

### After the demo / Sau khi demo
The server intentionally remains alive so you can send more requests. Stop it with scripts/stop.sh when finished.

---

### Tiếng Việt

Trạng thái server cuối cùng

Mục đích
Kết thúc demo bằng một kiểm tra vận hành rõ ràng thay vì giả định rằng có response thành công thì service chắc chắn vẫn khỏe.

Cell này làm gì
Cell gọi public status script, in các marker cuối của showcase và báo tổng số request đã hoàn thành.

Trạng thái kết thúc thành công
Hãy tìm SHOWCASE=PASS và COMPLETED_REQUESTS=11.

Server được giữ sống để bạn có thể thử thêm request. Khi hoàn tất, chạy scripts/stop.sh để dừng service.


In [ ]:
sh(['bash', 'scripts/status.sh'], cwd=REPO_DIR, env=env, check=True)
print('\nSHOWCASE=PASS')
print(f'COMPLETED_REQUESTS={len(SHOWCASE)}')
print('The server remains alive for additional experiments.')
print('When finished: bash scripts/stop.sh')
